In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

In [ ]:
# 1. Scaled Dot-Product Attention
# q: [B, H, T, Dh], k/v: [B, H, S, Dh]
def scaled_dot_product_attention(q, k, v, mask=None):
    assert q.size(-1) == k.size(-1), "Q/K 的 head_dim 必须相同"
    assert k.size(-2) == v.size(-2), "K/V 的序列长度必须相同"
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))  # [B,H,T,S]
    if mask is not None:
        assert torch.broadcast_shapes(scores.shape, mask.shape) == scores.shape
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    attn = torch.softmax(scores, dim=-1)  # 沿 key 维 S 做 softmax
    return attn @ v, attn

B, H, T, S, Dh = 2, 4, 5, 6, 8
q = torch.randn(B, H, T, Dh, device=device)
k = torch.randn(B, H, S, Dh, device=device)
v = torch.randn(B, H, S, Dh, device=device)
out, attn = scaled_dot_product_attention(q, k, v)
assert out.shape == (B, H, T, Dh)
assert torch.allclose(attn.sum(-1), torch.ones(B, H, T, device=device), atol=1e-5)

In [ ]:
# 7. 简化 MLA：缓存低秩 c_kv，而不是完整 K/V
class SimpleMLA(nn.Module):
    def __init__(self, d_model, num_heads, kv_lora_rank):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.w_dkv = nn.Linear(d_model, kv_lora_rank, bias=False)
        self.w_uk = nn.Linear(kv_lora_rank, d_model, bias=False)
        self.w_uv = nn.Linear(kv_lora_rank, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        c_kv = self.w_dkv(x)                              # [B,T,R]，推理时缓存它
        q = self.wq(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.w_uk(c_kv).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.w_uv(c_kv).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()[None, None]
        out, _ = scaled_dot_product_attention(q, k, v, mask)
        return self.wo(out.transpose(1, 2).contiguous().view(B, T, D)), c_kv

x = torch.randn(2, 5, 32, device=device)
mla = SimpleMLA(32, 4, kv_lora_rank=8).to(device)
mla_out, c_kv = mla(x)
assert mla_out.shape == x.shape and c_kv.shape == (2, 5, 8)